# 02 - Global transfer demand model training (Colab)

Thin runner notebook. **No training logic lives here** - this calls straight into `models/global_transfer/train_global.py`'s `train_and_save()` (already an importable function, no argparse/CLI to shell out to).

What this trains: one joint XGBoost demand regressor across the 2 real OBSERVED cities (NYC `zone_hourly_demand`, London `london_station_hourly_demand`), with a scaled static city-feature vector (`E_city`, from `models/global_transfer/build_features.py`) appended to the usual lag/EWMA temporal features. See that module's own docstring for the explicit honesty note this notebook does not repeat: with only 2 OBSERVED cities, this measures fit to those 2 cities, not validated cross-city generalization (`models/global_transfer/model_comparison.py` is the leave-one-city-out check, out of scope for this notebook).

No progressive sampling here: the joint city-hour dataset (`build_joint_dataset()`) is already small (a few thousand rows after per-city hourly aggregation, confirmed by a local dry run - see the agent's report), not a row-sampling problem.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Locate the repo and both warehouse files

This model needs **both** NYC and London DuckDB files (`data/warehouse/nyc_rides.duckdb`, 6.0 GB, and `data/warehouse/london_cycles.duckdb`, ~0.22 GB - both sizes verified locally). Set `GLOBAL_MOBILITY_DATA_ROOT` to a folder containing both files, same convention as notebook 01 (Drive or GCS, your choice - see `models/notebooks/README.md`).

In [ ]:
import os

REPO_URL = os.environ.get('GLOBAL_MOBILITY_REPO_URL', 'https://github.com/TeerthPurohit/Uber-nyc-TLC-Dataset.git')
REPO_DIR = '/content/nyc-tlc-repo'

os.environ.setdefault('GLOBAL_MOBILITY_DATA_ROOT', '/content/drive/MyDrive/global_mobility/repo')
DATA_ROOT = os.environ['GLOBAL_MOBILITY_DATA_ROOT']

if not os.path.isdir(REPO_DIR):
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    !git -C "$REPO_DIR" pull

print('repo:', REPO_DIR)
print('data root:', DATA_ROOT)

## 3. Install pinned dependencies (parity with local `requirements.txt`)

In [ ]:
!pip install -q -r "$REPO_DIR/requirements.txt"

## 4. Wire both DuckDB files into the expected repo paths

`models/global_transfer/train_global.py` hardcodes `NYC_DB` / `LONDON_DB` as `<repo_root>/data/warehouse/{nyc_rides,london_cycles}.duckdb` - symlink both from `GLOBAL_MOBILITY_DATA_ROOT` rather than patching the module.

In [ ]:
warehouse_dir = os.path.join(REPO_DIR, 'data', 'warehouse')
os.makedirs(warehouse_dir, exist_ok=True)

db_files = {}
for name in ('nyc_rides.duckdb', 'london_cycles.duckdb'):
    src = os.path.join(DATA_ROOT, name)
    dst = os.path.join(warehouse_dir, name)
    assert os.path.exists(src), (
        f'{name} not found at {src} -- set GLOBAL_MOBILITY_DATA_ROOT to the '
        'Drive/GCS folder that actually contains both warehouse files.'
    )
    if not os.path.exists(dst):
        os.symlink(src, dst)
    db_files[name] = dst
    print(f'{name} ready at {dst} ({os.path.getsize(dst) / 1e9:.2f} GB)')

## 5. Validate schema and row counts before training (fail loudly, not blind)

Checks the two source marts `models/global_transfer/train_global.py`'s `load_city_hourly_total()` actually queries, plus `global_cities` / `cities` / `worldmove_city_population` that `models/global_transfer/build_features.py::build_city_feature_table()` joins for the static city-feature vector.

In [ ]:
import duckdb

con = duckdb.connect(db_files['nyc_rides.duckdb'], read_only=True)
try:
    tables = {r[0] for r in con.execute("select table_name from information_schema.tables").fetchall()}
    for t in ('zone_hourly_demand', 'global_cities', 'cities', 'worldmove_city_population'):
        if t not in tables:
            raise RuntimeError(f'expected table {t!r} missing from nyc_rides.duckdb -- wrong/stale file?')
    n_demand = con.execute('select count(*) from zone_hourly_demand').fetchone()[0]
    n_cities = con.execute('select count(*) from global_cities').fetchone()[0]
    if n_demand < 1_000_000:
        raise RuntimeError(f'zone_hourly_demand has only {n_demand} rows, expected >= 1,000,000')
    if n_cities < 500:
        raise RuntimeError(f'global_cities has only {n_cities} rows, expected >= 500 (2 OBSERVED + ~522 TRANSFER)')
    print(f'nyc_rides.duckdb OK: zone_hourly_demand={n_demand:,} rows, global_cities={n_cities} rows')
finally:
    con.close()

con = duckdb.connect(db_files['london_cycles.duckdb'], read_only=True)
try:
    tables = {r[0] for r in con.execute("select table_name from information_schema.tables").fetchall()}
    if 'london_station_hourly_demand' not in tables:
        raise RuntimeError('expected table london_station_hourly_demand missing from london_cycles.duckdb')
    n_london = con.execute('select count(*) from london_station_hourly_demand').fetchone()[0]
    if n_london < 500_000:
        raise RuntimeError(f'london_station_hourly_demand has only {n_london} rows, expected >= 500,000')
    print(f'london_cycles.duckdb OK: london_station_hourly_demand={n_london:,} rows')
finally:
    con.close()

## 6. Train the joint global transfer model

Direct call into `train_and_save()` - builds the joint city-hour dataset (`build_joint_dataset()`), fits/tunes the small hyperparameter grid already defined in the script, refits on train+val, evaluates once on test. Saves `xgb_model.json` + `xgb_metadata.json` + `city_feature_scaler.json` into `models/global_transfer/`.

In [ ]:
import sys, time
sys.path.insert(0, REPO_DIR)
from models.global_transfer.train_global import train_and_save, demo

demo()  # smallest runnable self-check (block-gap detection, no leaked NaNs) before spending real training time

t0 = time.perf_counter()
meta = train_and_save()
elapsed_s = time.perf_counter() - t0

print(f"n_rows: {meta['n_rows']}")
print(f"val   RMSE={meta['metrics']['val_rmse']:.3f}  MAE={meta['metrics']['val_mae']:.3f}")
print(f"test  RMSE={meta['metrics']['test_rmse']:.3f}  MAE={meta['metrics']['test_mae']:.3f}")
print(f'training time: {elapsed_s:.1f}s')
print(meta['honesty_note'])

## 7. Copy artifacts back to the local repo

`scripts/refresh_model_registry.py` and `backend/services/model_service.py` key off these exact filenames under `models/global_transfer/` - copy them back to the same relative paths in your local checkout.

In [ ]:
import shutil

OUTPUT_DIR = f'{DATA_ROOT}/trained_artifacts'
os.makedirs(OUTPUT_DIR, exist_ok=True)

FILES_TO_COPY = [
    'models/global_transfer/xgb_model.json',
    'models/global_transfer/xgb_metadata.json',
    'models/global_transfer/city_feature_scaler.json',
]
for rel in FILES_TO_COPY:
    src = os.path.join(REPO_DIR, rel)
    dst_path = os.path.join(OUTPUT_DIR, rel.replace('/', '__'))
    shutil.copy(src, dst_path)

print(f'Copied {len(FILES_TO_COPY)} files to {OUTPUT_DIR}.')
print('Next: download these from Drive and place each one back at its original')
print('relative path under your local repo checkout, e.g.:')
for rel in FILES_TO_COPY:
    print(f'  {OUTPUT_DIR}/{rel.replace(chr(47), chr(95)+chr(95))}  ->  <local repo>/{rel}')
print()
print('Then locally: python scripts/refresh_model_registry.py')